## Multi-protocol CNN Raw CSI Experiments

This notebook runs the fixed-capacity CNN across block, LOVO, and cross-session protocols with one independent run per configured seed.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import torch

from utils.DL.dl_pipeline import (
    create_position_label_encoder,
    get_cache_path,
    get_results_path,
    prepare_dl_data,
    print_torch_environment,
    run_dl_experiments,
    show_dl_results,
)
from utils.config import DEFAULT_CNN_PARAMS, SEEDS


### Configuration

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = Path("/disco1500gb/Pedro_Data/")
CALIBRATION_MODE = "none"    # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": 60,
    "overlap_size": 30,
    "require_all_esps": False,
}

BANDS_TO_RUN = ("2.4 GHz", "5 GHz", "Fusion")
EXPECTED_SUBCARRIERS = {"2.4 GHz": 50, "5 GHz": 56}
EXPECTED_ANCHORS = {"2.4 GHz": 9, "5 GHz": 10}
ANCHOR_GROUPS = {
    "2.4 GHz": ["esp_01", "esp_02", "esp_03", "esp_04", "esp_05", "esp_07", "esp_08", "esp_09", "esp_10"],
    "5 GHz": ["esp_11", "esp_12", "esp_13", "esp_14", "esp_15", "esp_16", "esp_17", "esp_18", "esp_19", "esp_20"],
}

BLOCK_COUNT = 10
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42
FORCE_RETRAIN = False
SPLIT_MODES = ("block", "lovo", "cross_session")

CNN_PARAMS = {
    **DEFAULT_CNN_PARAMS,
    "model_label": "CNN",
    "epochs": 50,
    "patience": 15,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "n_blocks": BLOCK_COUNT,
    "anchor_groups": ANCHOR_GROUPS,
    "torch_version": torch.__version__,
}

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path()
plots_dir = results_dir / "plots"
for directory in (plots_dir, results_dir / "predictions", results_dir / "tables"):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")


Feature cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step30
Results path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/preproc=norm-none/feat=win60-step30


### Environment

In [3]:
DEVICE = print_torch_environment(require_cuda=True)


torch: 2.11.0+cu128
torch threads: 32
device: cuda
cuda device: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
Seeds: random=42, numpy=42, torch=42


### Data

In [4]:
processed_magnitude_data, feature_dataframes, csv_diagnostics, magnitude_summary = prepare_dl_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
)
display(magnitude_summary.head())


Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19


,scenario,location,user,esp,trial,samples,subcarriers,normalization,baseline_scope
0,1,C-1,06,16,01,2088,56,none,
1,1,C-1,06,15,01,1942,56,none,
2,1,C-1,06,09,01,1689,50,none,
3,1,C-1,06,07,01,1965,50,none,
4,1,C-1,06,04,01,1913,50,none,


In [5]:
for band in BANDS_TO_RUN:
    df = feature_dataframes[band]
    print(f"{band}: {df.shape[0]} windows, {df.shape[1]} columns")
    print(f"{band} dataframe hash: {pd.util.hash_pandas_object(df, index=True).sum()}")


[cache miss] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step30, computing...
2.4 GHz: 26913 windows, 2708 columns
2.4 GHz dataframe hash: 15954751541181741269
5 GHz: 28745 windows, 3368 columns
5 GHz dataframe hash: 14832925763032102390
Fusion: 26868 windows, 6068 columns
Fusion dataframe hash: 11223025498916198040


In [6]:
label_encoder = create_position_label_encoder(
    feature_dataframes,
    results_dir=results_dir,
    expected_classes=52,
)
print(label_encoder.classes_)


[CNN] label classes saved to /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/preproc=norm-none/feat=win60-step30/predictions/cnn_label_classes.json
['A-1' 'A-13' 'A-14' 'A-2' 'A-5' 'B-1' 'B-10' 'B-11' 'B-12' 'B-13' 'B-14'
 'B-2' 'B-5' 'B-8' 'C-1' 'C-10' 'C-11' 'C-14' 'C-2' 'C-3' 'C-4' 'C-5'
 'C-6' 'C-7' 'C-8' 'C-9' 'D-1' 'D-2' 'D-3' 'D-4' 'D-5' 'D-6' 'D-7' 'D-8'
 'E-1' 'E-10' 'E-11' 'E-12' 'E-13' 'E-2' 'E-3' 'E-4' 'E-5' 'E-6' 'E-7'
 'E-8' 'F-10' 'F-11' 'F-12' 'F-13' 'F-5' 'F-8']


### Train And Evaluate

In [ ]:
cnn_runs = run_dl_experiments(
    processed_magnitude_data,
    feature_dataframes,
    bands=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
    label_encoder=label_encoder,
    device=DEVICE,
    results_dir=results_dir,
    plots_dir=plots_dir,
    params=CNN_PARAMS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    expected_subcarriers=EXPECTED_SUBCARRIERS,
    expected_anchors=EXPECTED_ANCHORS,
    seeds=SEEDS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    val_size=VALIDATION_SIZE,
    force_retrain=FORCE_RETRAIN,
)



=== 2.4 GHz ===
[window arrays] cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/preproc=norm-none/feat=win60-step30/window_arrays/2_4ghz
[window arrays] 2.4 GHz: shape=(26913, 9, 50, 60), dtype=float16
[window arrays] 2.4 GHz anchors: ['esp_01', 'esp_02', 'esp_03', 'esp_04', 'esp_05', 'esp_07', 'esp_08', 'esp_09', 'esp_10']
2.4 GHz: anchors=9, subcarriers=50, windows=26913
WINDOW IDENTITY PASS: 2.4 GHz (26913 windows)
BLOCK SPLIT IDENTITY PASS: 2.4 GHz (train=16904, test=6147)
[CNN_small] 2.4 GHz: parameters=23188


/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/dl_pipeline.py:50: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)


[CNN_small] 2.4 GHz epoch 01/50: train_loss=3.9129 train_acc=0.0302 val_loss=3.8788 val_acc=0.0407 seconds=1.4
[CNN_small] 2.4 GHz epoch 02/50: train_loss=3.6963 train_acc=0.0698 val_loss=3.6629 val_acc=0.0793 seconds=0.9
[CNN_small] 2.4 GHz epoch 03/50: train_loss=3.4456 train_acc=0.1010 val_loss=4.8417 val_acc=0.0332 seconds=0.8
[CNN_small] 2.4 GHz epoch 04/50: train_loss=3.3213 train_acc=0.1204 val_loss=4.4080 val_acc=0.0830 seconds=0.8
[CNN_small] 2.4 GHz epoch 05/50: train_loss=3.2109 train_acc=0.1367 val_loss=3.5595 val_acc=0.1109 seconds=0.8
[CNN_small] 2.4 GHz epoch 06/50: train_loss=3.1298 train_acc=0.1513 val_loss=4.4908 val_acc=0.0723 seconds=0.8
[CNN_small] 2.4 GHz epoch 07/50: train_loss=3.0602 train_acc=0.1611 val_loss=5.0847 val_acc=0.0530 seconds=0.8
[CNN_small] 2.4 GHz epoch 08/50: train_loss=2.9981 train_acc=0.1729 val_loss=3.4768 val_acc=0.1521 seconds=0.8
[CNN_small] 2.4 GHz epoch 09/50: train_loss=2.9498 train_acc=0.1851 val_loss=4.3334 val_acc=0.0734 seconds=0.8
[

In [8]:
cnn_summary, test_accuracy_comparison = show_dl_results(results_dir)
display(cnn_summary)

display(test_accuracy_comparison)


,dataset,model,split,position_accuracy,macro_f1,room_accuracy,mean_distance_error,median_distance_error,rmse_distance_error,p90_distance_error,...,fit_seconds,predict_seconds,wall_seconds,used_estimator,parameter_count,best_val_accuracy,best_epoch,stopped_epoch,patience_triggered,mean_seconds_per_epoch
0,2.4 GHz,CNN_small,block,0.334960,0.326971,0.795510,2.288231,1.414214,3.428581,5.385165,...,42.613989,0.265169,42.879158,DualBandCNN,23188.0,0.356722,45.0,50.0,False,0.851844
1,5 GHz,CNN_small,block,0.325228,0.321073,0.893630,1.868486,1.414214,2.690594,4.123106,...,62.562475,0.462743,63.025218,DualBandCNN,23332.0,0.342389,45.0,50.0,False,1.250831
2,Fusion,CNN_small,block,0.494051,0.491111,0.915566,1.403991,1.000000,2.411903,4.123106,...,92.015048,0.823535,92.838583,DualBandCNN,39684.0,0.506186,46.0,50.0,False,1.839640


,dataset,model,position_accuracy,parameter_count
0,2.4 GHz,CNN_small,0.334960,23188.0
1,5 GHz,CNN_small,0.325228,23332.0
2,Fusion,CNN_small,0.494051,39684.0
